### Importing packages

In [1]:
# importing imp packages
import os
import zipfile
import random
import yaml
from google.cloud import storage
import uuid
from datetime import date

### From training.ipynb

In [2]:
def download_data_from_gcs(bucket_name: str, gcs_path: str, local_dir: str) -> None:
    """
    Downloads a directory and its contents from a Google Cloud Storage bucket.

    Args:
        bucket_name (str): The name of the GCS bucket.
        gcs_path (str): The path to the directory in GCS (e.g., "datasets/my_data/").
        local_dir (str): The local directory to save the downloaded files.
    """
    print(
        f"Attempting to download data from gs://{bucket_name}/{gcs_path} to {local_dir}"
    )

    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)

        # Ensure local directory exists
        os.makedirs(local_dir, exist_ok=True)

        blobs = bucket.list_blobs(
            prefix=gcs_path
        )  # List all blobs with the given prefix
        downloaded_count = 0
        for blob in blobs:
            # Skip blobs that represent directories themselves (ending with '/')
            # And filter for files that end with '.zip'
            if not blob.name.endswith("/") and blob.name.endswith(".zip"):
                # Construct local file path, preserving relative directory structure if any
                local_file_name = os.path.basename(blob.name)
                local_file_path = os.path.join(local_dir, local_file_name)

                blob.download_to_filename(local_file_path)
                downloaded_count += 1
            else:
                print(f"Skipping non-zipped file or directory: {blob.name}")

        if downloaded_count == 0:
            print(
                f"No zipped files found or downloaded from gs://{bucket_name}/{gcs_path}. "
                f"Please check bucket name and GCS path, and ensure there are .zip files present."
            )
        else:
            print(f"Successfully downloaded {downloaded_count} zipped files from GCS.")

    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"Error downloading data from GCS: {e}")
        print("Please ensure your Google Cloud credentials are set up correctly.")
        print(
            "You can use `gcloud auth application-default login` for local development or set the "
            "`GOOGLE_APPLICATION_CREDENTIALS` environment variable for service accounts."
        )
        exit(1)  # Exit if data download fails


# unzipping files


def unzipDataset(folderPath: str) -> None:
    """
    Unzips all .zip files in the specified folder.
    Args:
        folderPath (str): The path to the folder containing .zip files.
    """
    files = os.listdir(folderPath)
    for file in files:
        fullPath = os.path.join(folderPath, file)
        # Check if the item is a file and ends with .zip
        if os.path.isfile(fullPath) and fullPath.endswith(".zip"):
            filename = file.split(".")[0]
            try:
                with zipfile.ZipFile(fullPath, "r") as zip_ref:
                    # Extract to a subdirectory with the same name as the zip file
                    extract_dir = os.path.join(folderPath, filename)
                    print(f"Unzipping {file} into {extract_dir}")
                    zip_ref.extractall(extract_dir)

                # Remove the zip file after successful extraction
                os.remove(fullPath)
            except Exception as e:
                print(f"Error unzipping or removing file {fullPath}: {e}")

### Configurations

In [3]:
GCS_BUCKET_NAME = "open-cityvision"
GCS_DATA_PATH = "Dataset/"  # Path to the zipped files in GCS (e.g., 'raw_data/')

LOCAL_DATA_DIR = "yolo_dataset/"  # Local directory to download and process files


# Define the split ratios for your dataset.
TRAIN_SPLIT_RATIO = 0.8
VAL_SPLIT_RATIO = 0.1
TEST_SPLIT_RATIO = 0.1

### New function

In [4]:
def get_class_names_from_yaml(local_dir: str) -> list:
    """
    Finds and extracts class names from the first data.yaml file found.
    """
    for root, _, files in os.walk(local_dir):
        if "data.yaml" in files:
            yaml_path = os.path.join(root, "data.yaml")
            print(f"Found data.yaml at {yaml_path}. Reading class names...")
            with open(yaml_path, "r") as f:
                data = yaml.safe_load(f)
                if "names" in data:
                    print("Successfully extracted class names.")
                    return data["names"]

    print(
        "Warning: No 'data.yaml' file found to extract class names. Returning an empty list."
    )
    return []


def combine_and_split_dataset(source_dir):
    """
    Creates a combined folder with train, val, and test subfolders,
    assigning the largest source subdirectory to train, and the others to val/test.
    This version ignores the ratio parameters and splits by source directory.
    Args:
        source_dir (str): The directory containing the source datasets.
    Returns:
        str: The path to the newly created combined dataset directory.


    """
    print("Combining based on folder size")

    # Step 1: Identify all top-level subdirectories (potential source datasets)
    source_subdirs = []
    for d in os.listdir(source_dir):
        if os.path.isdir(os.path.join(source_dir, d)):
            source_subdirs.append(os.path.join(source_dir, d))
    if len(source_subdirs) < 3:
        print(
            "Error: Need at least 3 subdirectories in source_dir for Train/Val/Test split."
        )
        return

    # Dictionary to store all image paths and their corresponding label path for each source folder
    folder_data = {}

    # Collect data and count image-label pairs for each subdirectory
    for subdir in source_subdirs:
        # Assuming the structure is: subdir/images/... and subdir/labels/...

        subdir_name = os.path.basename(subdir)
        folder_data[subdir_name] = {
            "image_paths": [],
            "label_map": {},  # map image_path -> [label_root, label_path]
            "count": 0,
        }

        # Recursively walk through the 'images' subdirectory
        # search for images folder

        for root, dirs, files in os.walk(subdir):
            for file in files:
                # We are looking for image files (e.g., .jpg, .png)
                if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".gif")):
                    image_path = os.path.join(root, file)
                    # Determine expected label path
                    # This relies on the "images" folder being directly replaced by "labels"
                    label_root = root.replace("images", "labels", 1)
                    label_path = os.path.join(label_root, file.replace(".jpg", ".txt"))

                    # Check if the label file exists
                    if not os.path.exists(label_path):
                        label_dir = os.path.dirname(label_path)
                        # Check if the directory exists and create it recursively if it doesn't
                        if not os.path.exists(label_dir):
                            print(f"Creating directory: {label_dir}")
                            os.makedirs(
                                label_dir, exist_ok=True
                            )  # exist_ok=True prevents an error if the directory already exists
                            # ------------------------
                        with open(label_path, "w") as f:
                            pass
                        # only append the image path if the label exists
                    folder_data[subdir_name]["image_paths"].append(image_path)
                    folder_data[subdir_name]["label_map"][image_path] = [
                        label_root,
                        label_path,
                    ]
                    folder_data[subdir_name]["count"] += 1

    # Get folder counts and sort them
    folder_counts = {
        name: data["count"] for name, data in folder_data.items() if data["count"] > 0
    }
    for name, data in folder_data.items():
        if data["count"] > 0:
            folder_counts[name] = data["count"]
    if len(folder_counts) < 3:
        print(
            "Error: Not enough subdirectories with valid image/label pairs (minimum 3 required)."
        )
        return

    sorted_folders = sorted(
        folder_counts.items(), key=lambda item: item[1]
    )  # Sort by count

    # Step 4: Assign smallest two folders to val and test
    # The two smallest (by count) folders are the first two elements in sorted_folders
    val_folder_name, _ = sorted_folders[0]
    test_folder_name, _ = sorted_folders[1]

    # The rest are assigned to train
    train_folder_names = [name for name, _ in sorted_folders[2:]]

    # Group all file paths based on the new folder split
    train_images = []
    val_images = []
    test_images = []
    all_label_paths = (
        {}
    )  # Centralized map for all image_path -> [label_root, label_path]

    for name, data in folder_data.items():
        all_label_paths.update(data["label_map"])  # Merge all label maps

        if name in train_folder_names:
            train_images.extend(data["image_paths"])
        elif name == val_folder_name:
            val_images.extend(data["image_paths"])
        elif name == test_folder_name:
            test_images.extend(data["image_paths"])

    # Define new directory structure
    total_images = len(train_images) + len(val_images) + len(test_images)
    Today = date.today()
    output_dir = os.path.join(source_dir, f"{Today:%Y-%m-%d}_len_{total_images}")
    # create output directory if doesnt exist
    os.makedirs(output_dir, exist_ok=True)
    train_images_dir = os.path.join(output_dir, "train", "images")
    train_labels_dir = os.path.join(output_dir, "train", "labels")
    val_images_dir = os.path.join(output_dir, "val", "images")
    val_labels_dir = os.path.join(output_dir, "val", "labels")
    test_images_dir = os.path.join(output_dir, "test", "images")
    test_labels_dir = os.path.join(output_dir, "test", "labels")

    # Create directories
    for d in [
        train_images_dir,
        train_labels_dir,
        val_images_dir,
        val_labels_dir,
        test_images_dir,
        test_labels_dir,
    ]:
        os.makedirs(d, exist_ok=True)

    # Helper function to move files with unique filenames
    def move_files(image_paths, image_dest, label_dest, all_label_paths):
        """
        Moves image and corresponding label files, creates a unique filename
        based on the source directory structure, and updates the label file paths.

        Args:
            image_paths (list): List of original absolute image paths.
            image_dest (str): Destination directory for images (e.g., 'output/train/images').
            label_dest (str): Destination directory for labels (e.g., 'output/train/labels').
            all_label_paths (dict): A map from original image path to [label_root, label_path].

        Returns:
            list: List of new relative paths for the moved images.
        """
        new_img_paths = []
        for img_path in image_paths:
            # Retrieve the label paths from the central map
            label = all_label_paths.get(img_path, None)
            label_root, label_path = label if label else (None, None)

            # Get the extension of the image file (not strictly needed here, but harmless)
            _, ext = os.path.splitext(os.path.basename(img_path))

            # Check if corresponding label file exists before moving
            if label_path and os.path.exists(label_path):
                # edit the label file to remove the 6th column if it exists
                edit_label_file(label_path)

                # Generate a unique name for the file using the parent directory name and the filename
                # e.g., 'path/to/SCU3I3_202405090345_003/image_1.jpg' -> 'SCU3I3_202405090345_003_image_1.jpg'
                image_path_list = img_path.replace("\\", "/").split(os.path.sep)
                try:
                    unique_filename = f"{image_path_list[-2]}_{image_path_list[-1]}"
                except IndexError:
                    # generate a random uuid as filename when the path structure is unexpected
                    unique_filename = f"{uuid.uuid4()}{ext}"

                # Get parts of the destination directory for the relative path
                # e.g., 'yolo_dataset/combined/train/images' -> ['train', 'images']
                imgNew_path_list = image_dest.replace("\\", "/").split(os.path.sep)
                print(
                    f"Moving {img_path} to {os.path.join(image_dest, unique_filename)}"
                )
                print("imgNew_path_list:", imgNew_path_list)
                # Construct the new relative path for the output text file (e.g., './train/images/unique_file.jpg')
                new_img_paths.append(
                    os.path.join(
                        ".", imgNew_path_list[-2], imgNew_path_list[-1], unique_filename
                    ).replace("\\", "/")
                )

                # Rename and move the image file
                # os.rename performs a move operation if the destination is on the same file system
                os.rename(img_path, os.path.join(image_dest, unique_filename))

                # Rename and move the label file to match the new image name
                os.rename(
                    label_path,
                    os.path.join(
                        label_dest, os.path.splitext(unique_filename)[0] + ".txt"
                    ),
                )
            else:
                print(f"Warning: Label file for {img_path} not found. Skipping.")

        return new_img_paths

    def edit_label_file(label_path):
        """
        Edits the label file to remove the 6th column if it existis"""
        if not os.path.exists(label_path):
            print(f"Warning: Label file {label_path} does not exist. Skipping.")
            return
        with open(label_path, "r") as f:
            lines = f.readlines()
        with open(label_path, "w") as f:
            for line in lines:
                parts = line.strip().split(sep=" ")
                if len(parts) > 5:
                    # Remove the 6th column (index 5)
                    parts.pop()
                f.write(" ".join(parts) + "\n")

    def create_text_file(file_list, output_path):
        """
        Creates a text file listing all file paths in the provided list.

        Args:
            file_list (list): List of file paths to include in the text file.
            output_path (str): Path to save the generated text file.

        """
        with open(output_path, "w") as f:
            for file_path in file_list:
                f.write(f"{file_path}\n")

    print(f"Moving {len(train_images)} files to train set.")
    new_train_paths = move_files(
        train_images, train_images_dir, train_labels_dir, all_label_paths
    )
    print(f"Moving {len(val_images)} files to validation set.")
    new_val_paths = move_files(
        val_images, val_images_dir, val_labels_dir, all_label_paths
    )
    print(f"Moving {len(test_images)} files to test set.")
    new_test_paths = move_files(
        test_images, test_images_dir, test_labels_dir, all_label_paths
    )
    # writing the text files
    print("Writing train.txt, val.txt, and test.txt files...")
    create_text_file(new_train_paths, os.path.join(output_dir, "train.txt"))
    create_text_file(new_val_paths, os.path.join(output_dir, "val.txt"))
    create_text_file(new_test_paths, os.path.join(output_dir, "test.txt"))
    print("Dataset combination and splitting complete.")
    return output_dir


print(f"Current Working Directory (CWD): {os.getcwd()}")


def create_data_yaml(output_dir, class_names):
    """
    Creates the data.yaml file required for YOLOv8 training.
    """
    path = output_dir.replace("/", os.path.sep).split(os.path.sep)[
        -1
    ]  # Get the last part of the output directory path
    data = {
        "path": f"{path}",
        "train": "train.txt",
        "val": "val.txt",
        "test": "test.txt",
        "nc": len(class_names),
        "names": class_names,
    }

    yaml_file_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_file_path, "w") as f:
        yaml.dump(data, f, sort_keys=False)

    return path


def upload_combined_dataset(local_dir, bucket_name, destination_prefix):
    """
    Uploads the entire processed dataset folder to GCS.
    """
    print(
        f"Uploading combined dataset to GCS bucket: {bucket_name}, prefix: {destination_prefix}..."
    )
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    # Upload a zipped version of the dataset
    # Create a zip file of the local directory
    zip_file_path = f"{destination_prefix}.zip"
    with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(local_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Create a relative path for the zip file
                relative_path = os.path.relpath(file_path, local_dir)
                zipf.write(file_path, relative_path)
    # Upload the zip file to GCS
    blob = bucket.blob(zip_file_path)
    blob.upload_from_filename(zip_file_path)
    print(f"Uploaded {zip_file_path} to gs://{bucket_name}/{destination_prefix}")
    for root, _, files in os.walk(local_dir):
        for file in files:
            local_file_path = os.path.join(root, file)
            # Create a GCS destination path that maintains the folder structure
            relative_path = os.path.relpath(local_file_path, local_dir)
            gcs_path = os.path.join(destination_prefix, relative_path).replace(
                "\\", "/"
            )  # Use forward slashes

            blob = bucket.blob(gcs_path)
            blob.upload_from_filename(local_file_path)

    print("Upload complete.")

Current Working Directory (CWD): c:\Users\MJATTU\Desktop\Projects\cityvision\notebooks


In [5]:
os.path.sep

'\\'

### Main Code

In [6]:
if __name__ == "__main__":
    # clean the folders if they exist
    if os.path.exists(LOCAL_DATA_DIR):
        print(f"Cleaning up existing local directory: {LOCAL_DATA_DIR}")
        for root, dirs, files in os.walk(LOCAL_DATA_DIR, topdown=False):
            for name in files:
                os.remove(os.path.join(root, name))
            for name in dirs:
                os.rmdir(os.path.join(root, name))
        os.rmdir(LOCAL_DATA_DIR)

    # Step 1: Download from GCS using the new function
    download_data_from_gcs(GCS_BUCKET_NAME, GCS_DATA_PATH, LOCAL_DATA_DIR)

    # Step 2: Unzip the downloaded files using the new function
    unzipDataset(LOCAL_DATA_DIR)

    # Step 3: Get class names from a data.yaml file
    class_names = get_class_names_from_yaml(LOCAL_DATA_DIR)

    # Step 4: Combine and split the dataset
    combined_dataset_output = combine_and_split_dataset(LOCAL_DATA_DIR)

    # Step 5: Create the data.yaml file with the extracted class names
    path = create_data_yaml(combined_dataset_output, class_names)
    DESTINATION_PREFIX = path.split("\\")[-1]
    # Step 6: Upload the new dataset to GCS
    upload_combined_dataset(
        combined_dataset_output, GCS_BUCKET_NAME, DESTINATION_PREFIX
    )

    print("\nDataset preparation and upload process finished.")

Attempting to download data from gs://open-cityvision/Dataset/ to yolo_dataset/
Skipping non-zipped file or directory: Dataset/


KeyboardInterrupt: 

### Configuring GCLOUD for the Jupyter Notebook

In [ ]:
import os

print(os.environ["PATH"])

# run this command on the terminal and compare the output with the notebook - "which gcloud"

/home2/mikjat/projects/newdir/cityvision/ven/bin:/home2/mikjat/.vscode-server/cli/servers/Stable-0f0d87fa9e96c856c5212fc86db137ac0d783365/server/bin/remote-cli:/home2/mikjat/.local/bin:/home2/mikjat/projects/newdir/cityvision/google-cloud-sdk/bin:/home2/mikjat/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin


In [ ]:
# adding the gcloud command to the PATH
import os

os.environ["PATH"] += (
    os.pathsep + "/home2/mikjat/projects/newdir/cityvision/google-cloud-sdk/bin"
)

In [ ]:
# checking if the gcloud command works
!gcloud --version

Google Cloud SDK 533.0.0
bq 2.1.22
bundled-python3-unix 3.12.9
core 2025.08.01
gcloud-crc32c 1.0.0
gsutil 5.35
Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update


In [ ]:
# Cell 1: Set the environment variable and then authenticate

import os
import google.auth

# Use os.path.expanduser() to correctly expand the "~" character
credentials_path = os.path.expanduser(
    "~/.config/gcloud/application_default_credentials.json"
)

# Check if the file exists before setting the environment variable
if os.path.exists(credentials_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
    print(
        f"GOOGLE_APPLICATION_CREDENTIALS environment variable set to: {credentials_path}"
    )

    # Now, attempt to authenticate
    try:
        credentials, project = google.auth.default()
        print("Authenticated with Google Cloud successfully.")
        print(f"Using credentials for project: {project}")
    except Exception as e:
        print(f"Authentication failed: {e}")
        # This will give a more descriptive error if authentication fails for other reasons.
else:
    print(f"Error: Credentials file not found at {credentials_path}")
    print(
        "Please ensure you have run 'gcloud auth application-default login' in your terminal."
    )

Error: Credentials file not found at C:\Users\MJATTU/.config/gcloud/application_default_credentials.json
Please ensure you have run 'gcloud auth application-default login' in your terminal.
